In [ ]:
import os
import re
import anthropic
from dotenv import load_dotenv
from openai import OpenAIError
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_xai import ChatXAI
from langchain.prompts import PromptTemplate
from datasets import load_dataset
from langchain.llms.base import BaseLLM
from langchain.chat_models.base import BaseChatModel
from langchain.schema import HumanMessage
from pydantic import BaseModel, Field
import multiprocessing
from langchain.output_parsers import PydanticOutputParser
from functools import partial
import time
from typing import List, Dict
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

load_dotenv()

In [ ]:
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

# Define LLMs with API keys loaded from environment variables
llms = {
    "chatgpt": ChatOpenAI(
        model_name="gpt-4o",
        openai_api_key=os.getenv("OPENAI_API_KEY")
    ),
    "claude": ChatAnthropic(
        model_name="claude-3-sonnet-20241022",
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
    ),
    "deepseek": ChatDeepSeek(
        model="deepseek-chat",
        api_key=os.getenv("DEEPSEEK_API_KEY")
    )
}

In [ ]:
# Define the output model for LangChain to parse the response
class CodeBlock(BaseModel):
    id: str = Field(description="The ID of the block to be modified")
    code: str = Field(description="The full code of the block after modification")

class ModificationResult(BaseModel):
    modifications: List[CodeBlock] = Field(description="List of ONLY the code blocks that were modified")

# Function to split file content into blocks of ~30 lines
def split_into_blocks(file_content):
    lines = file_content.split('\n')
    blocks = {}
    block_id = 1
    
    for i in range(0, len(lines), 30):
        block_lines = lines[i:i+30]
        block_key = f"block{block_id}"
        blocks[block_key] = '\n'.join(block_lines)
        block_id += 1
    
    return blocks

# Function to prepare file content with block markers
def prepare_file_with_blocks(file_content):
    blocks = split_into_blocks(file_content)
    formatted_content = []
    
    for block_id, block_content in blocks.items():
        formatted_content.append(f"# [START {block_id}]")
        formatted_content.append(block_content)
        formatted_content.append(f"# [END {block_id}]")
    
    return '\n'.join(formatted_content), blocks

# Create a parser for the output
parser = PydanticOutputParser(pydantic_object=ModificationResult)

In [ ]:
# Updated Prompt Template with JSON output format
prompt_template = PromptTemplate(
    input_variables=["problem_statement", "file_content", "parser_instructions"],
    template="""
We are solving the following issue:
--- BEGIN ISSUE ---
{problem_statement}
--- END ISSUE ---

Below is the relevant file, split into blocks:
--- BEGIN FILE ---
```
{file_content}
```
--- END FILE ---

Please fix the issues in the code by modifying the appropriate blocks. Provide your response in JSON format.

{parser_instructions}

IMPORTANT INSTRUCTIONS:
1. ONLY include blocks that you actually modified in your response.
2. Do NOT include unchanged blocks in your response.
3. For each modified block, include:
   - The ID of the block (e.g., "block1", "block2", etc.)
   - The complete code for that block after your modifications
4. Make minimal changes necessary to fix the issue.
5. Your response must be valid JSON that can be parsed according to the format above.
"""
)

def extract_modified_file_path(patch):
    """Extracts the modified file path from the first line of a Git diff."""
    match = re.search(r'diff --git a/(.*?) b/', patch)
    return match.group(1) if match else None

def generate_diff(original_blocks, modified_blocks, file_path):
    """Generates a unified diff from original and modified blocks."""
    diff_lines = [
        f"--- a/{file_path}",
        f"+++ b/{file_path}"
    ]
    
    # Process each modified block
    for mod_block in modified_blocks:
        block_id = mod_block.id
        if block_id not in original_blocks:
            print(f"[Warning] Block {block_id} not found in original blocks")
            continue
        
        original_content = original_blocks[block_id]
        modified_content = mod_block.code
        
        # Skip if no changes
        if original_content == modified_content:
            continue
            
        # Get line numbers for the block
        original_lines = original_content.split('\n')
        modified_lines = modified_content.split('\n')
        
        # Calculate line numbers based on block position
        # This is a simplified approach - in a real implementation, you'd need to track actual line numbers
        line_number = list(original_blocks.keys()).index(block_id) * 30 + 1
        
        # Add hunk header
        diff_lines.append(f"@@ -{line_number},{len(original_lines)} +{line_number},{len(modified_lines)} @@")
        
        # Add lines with prefixes
        for line in original_lines:
            diff_lines.append(f"-{line}")
        for line in modified_lines:
            diff_lines.append(f"+{line}")
    
    return '\n'.join(diff_lines)

def process_task(task, llm_name):
    """Processes a single task using the specified LLM."""
    try:
        instance_id = task["instance_id"]
        problem_statement = task["problem_statement"]
        patch = task["patch"]
        
        # Extract file path from patch
        file_path = extract_modified_file_path(patch)
        if not file_path:
            print(f"[Warning] No file path found for {instance_id}")
            return
        
        # Read file content
        file_full_path = f"./codebases/{instance_id}/{file_path}"
        if not os.path.exists(file_full_path):
            print(f"[Error] File not found: {file_full_path}")
            return
        
        with open(file_full_path, "r", encoding="utf-8") as f:
            file_content = f.read()
        
        # Split file content into blocks
        formatted_file_content, original_blocks = prepare_file_with_blocks(file_content)
        
        # Format prompt with LangChain parser instructions
        prompt = prompt_template.format(
            problem_statement=problem_statement, 
            file_content=formatted_file_content,
            parser_instructions=parser.get_format_instructions()
        )
        
        # Get response from LLM
        llm = llms[llm_name]
        if isinstance(llm, BaseChatModel):
            response = llm.invoke([HumanMessage(content=prompt)]).content
        elif isinstance(llm, BaseLLM):
            response = llm.predict(prompt)
        else:
            raise ValueError(f"Unknown LLM type for {llm_name}")
        
        # Parse the response
        try:
            # Extract JSON from the response if needed
            json_match = re.search(r'```json\s*(.*?)\s*```', response, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
            else:
                json_str = response
                
            # Parse the JSON response
            parsed_response = parser.parse(json_str)
            
            # Generate the diff
            diff_output = generate_diff(original_blocks, parsed_response.modifications, file_path)
            
            # Save the diff
            output_path = f"./test_outputs/json_format/{instance_id}_{llm_name}.diff"
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(diff_output)
            
            print(f"[Success] Diff saved: {output_path}")
            
            # Also save the raw JSON response for debugging
            json_output_path = f"./test_outputs/{llm_name}/json_format/{instance_id}.json"
            with open(json_output_path, "w", encoding="utf-8") as f:
                f.write(json_str)
                
        except Exception as e:
            print(f"[Error] Failed to parse response for {instance_id} with {llm_name}: {e}")
            # Save raw response for debugging
            output_path = f"./test_outputs/{llm_name}/json_format/{instance_id}.raw"
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(response)
            
    except Exception as e:
        print(f"[Error] Failed to process {instance_id} with {llm_name}: {e}")

In [ ]:
# Iterate through dataset and process each task
llm_name = 'chatgpt'
for task in dataset:
    process_task(task, llm_name)